# Inside a neuron: what the playground does not show

In WK3 you saw data go in and a decision boundary come out. Here we open the box, with 2D inputs so that **every step can be drawn**.

$$\text{vector} \;\rightarrow\; W x \;\rightarrow\; +\,b \;\rightarrow\; \phi(\cdot) \;\rightarrow\; \text{decision}$$

In each part: **first guess** what the picture will look like, then run the cell.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

activations = {
    "identity": lambda z: z,
    "tanh":     np.tanh,
    "sigmoid":  lambda z: 1 / (1 + np.exp(-z)),
    "relu":     lambda z: np.maximum(0, z),
    "leaky relu": lambda z: np.where(z > 0, z, 0.1 * z),
    "swish":    lambda z: z / (1 + np.exp(-z)),        # PyTorch: nn.SiLU
    "gelu":     lambda z: 0.5 * z * (1 + np.vectorize(math.erf)(z / math.sqrt(2))),
    "abs":      np.abs,
    "bump":     lambda z: np.exp(-z**2),
}

def arrow(ax, v, color, label, start=(0, 0)):
    ax.annotate("", xy=np.add(start, v), xytext=start,
                arrowprops=dict(arrowstyle="->", color=color, lw=2.5))
    ax.text(*(np.add(start, v) * 1.08), label, color=color, fontsize=13)

def square(ax, title, lim=4):
    ax.set(xlim=(-lim, lim), ylim=(-lim, lim), title=title, aspect="equal")
    ax.axhline(0, color="k", lw=.5); ax.axvline(0, color="k", lw=.5)

## 1. A vector, and the basis you describe it in

A vector is an arrow. The numbers are a **recipe**: how much of each basis vector you need.

$$x = x_1 e_1 + x_2 e_2$$

Pick another basis $b_1, b_2$: the arrow stays, the numbers change. We find them by solving $B c = x$, where the columns of $B$ are $b_1, b_2$.

In [ ]:
x  = np.array([2.0, 3.0])
b1 = np.array([1.0, 1.0])      # change me
b2 = np.array([-1.0, 1.0])     # change me (what if b2 = 2 * b1 ?)

c = np.linalg.solve(np.column_stack([b1, b2]), x)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (u, v, k, names) in zip(axes, [(np.array([1, 0]), np.array([0, 1]), x, ("e1", "e2")),
                                       (b1, b2, c, ("b1", "b2"))]):
    square(ax, f"x = {k[0]:.2f} {names[0]} + {k[1]:.2f} {names[1]}")
    arrow(ax, k[0]*u, "tab:blue", "")
    arrow(ax, k[1]*v, "tab:green", "", start=k[0]*u)
    arrow(ax, u, "navy", names[0]); arrow(ax, v, "darkgreen", names[1])
    arrow(ax, x, "tab:red", "x")
plt.show()

## 2. A matrix moves the basis, and everything follows

The **columns of $W$** are where $e_1$ and $e_2$ land. Since $Wx = x_1 (W e_1) + x_2 (W e_2)$, that fixes where every point goes. Grid lines stay straight, parallel and evenly spaced.

In [ ]:
t = np.linspace(-2, 2, 200)
lines  = [np.stack([t, np.full_like(t, k)]) for k in np.linspace(-2, 2, 9)]
lines += [L[::-1] for L in lines]
colors = ["tab:blue"]*9 + ["tab:green"]*9

def draw_grid(ax, f, title):
    square(ax, title)
    for L, col in zip(lines, colors):
        ax.plot(*f(L), color=col, lw=1)

W = np.array([[1.0, 0.8],
              [0.2, 1.2]])     # try a rotation, a scaling, and [[1, 2], [0.5, 1]]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
draw_grid(axes[0], lambda L: L, "input space")
draw_grid(axes[1], lambda L: W @ L, f"after W   (det = {np.linalg.det(W):.2f})")
for ax, M_ in zip(axes, [np.eye(2), W]):
    arrow(ax, M_[:, 0], "navy", "e1"); arrow(ax, M_[:, 1], "darkgreen", "e2")
plt.show()

## Meet the activation functions

An activation $\phi$ takes **one number** and returns one number. Look at its shape first: where is it flat? where is it steep? does it ever go **down**?

The dashed line is the slope. Training uses it: where the slope is near 0, the weights barely learn. `swish` and `gelu` are newer, smooth relatives of `relu`.

In [ ]:
z = np.linspace(-5, 5, 1001)
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, (name, phi) in zip(axes.flat, activations.items()):
    ax.plot(z, phi(z), lw=2.5, label="function")
    ax.plot(z, np.gradient(phi(z), z), "--", color="tab:orange", label="slope")
    ax.set(title=name, ylim=(-1.5, 3))
    ax.axhline(0, color="k", lw=.5); ax.axvline(0, color="k", lw=.5)
axes[0, 0].legend()
plt.tight_layout(); plt.show()

## 3. One neuron

$$y = \phi(w \cdot x + b)$$

$w \cdot x$ asks: **how much does $x$ point along $w$?** (the inner product you already know). $b$ shifts the threshold. $\phi$ re-labels the result.

Before running: if you change $\phi$, does the boundary bend?

In [ ]:
w = np.array([1.5, 1.0])   # change me
b = -1.0                   # change me

xx, yy = np.meshgrid(np.linspace(-4, 4, 300), np.linspace(-4, 4, 300))
z = w[0]*xx + w[1]*yy + b

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for ax, (name, phi) in zip(axes.flat, activations.items()):
    out = phi(z)
    level = phi(np.array(1.0))        # boundary: output equals phi(1), i.e. where z = 1
    square(ax, f"{name}: boundary at y = {level:.2f}")
    ax.contourf(xx, yy, out, 30, cmap="coolwarm")
    ax.contour(xx, yy, out, [level], colors="k", linewidths=2)
    arrow(ax, w, "k", "w")
plt.tight_layout(); plt.show()

**What do you see?**

- identity, tanh, sigmoid, relu, leaky relu, swish, gelu: the colours differ, the boundary is the **same straight line**, perpendicular to $w$. One neuron with an always-increasing (or non-decreasing) $\phi$ cannot bend it.
- swish and gelu dip slightly below zero, so at some **negative** levels they also give two lines.
- abs and bump go up *and* down, so you get **two** lines. Still straight.

So where do curved boundaries come from?

## 4. Two neurons: now we can watch space being warped

Two neurons turn a 2D point into a new 2D point: $h = \phi(Wx + b)$. Each row of $W$ is one neuron.

In [ ]:
def warp(name, W, b):
    phi = activations[name]
    steps = [("1. input x",  lambda L: L),
             ("2. W x",      lambda L: W @ L),
             ("3. W x + b",  lambda L: W @ L + b[:, None]),
             (f"4. {name}(W x + b)", lambda L: phi(W @ L + b[:, None]))]
    fig, axes = plt.subplots(1, 4, figsize=(17, 4))
    for ax, (title, f) in zip(axes, steps):
        draw_grid(ax, f, title)
    plt.show()

W = np.array([[1.0, 0.8],
              [0.2, 1.2]])
b = np.array([0.5, -0.5])

for name in ["identity", "tanh", "sigmoid", "relu"]:   # also try "abs", "bump"
    warp(name, W, b)

- Which steps keep lines straight? Which one does not?
- Where does `relu` fold or flatten space? Can you get the original point back?
- Multiply `W` by 5. What happens with `tanh`?
- Set `W = [[1, 2], [0.5, 1]]`. What is lost, even before $\phi$?

## 5. Put it together: bend space, then cut it with a straight line

The network has **one hidden layer** and **one output layer**. The input is not counted as a layer: it is just the data.

| | what goes in → out | computation | learned numbers |
|---|---|---|---|
| input | a point $x = (x_1, x_2)$ | none | 0 |
| hidden layer (2 neurons) | $x \rightarrow h = (h_1, h_2)$ | $h = \tanh(Wx + b)$ | 4 weights + 2 biases |
| output layer (1 neuron) | $h \rightarrow$ one score | $s = w \cdot h + c$ | 2 weights + 1 bias |

score $s > 0$: red class, score $s < 0$: blue class. The **boundary** is where $s = 0$.

**How to read the three pictures** (same points, same colours):

1. **Input:** the data and a square grid, before the network does anything.
2. **Hidden space:** every point and grid line is moved to its new position $h$. The axes are now the outputs of the two hidden neurons. Here the output neuron cuts with a **straight line**.
3. **Back to input:** colour every original point $x$ by the decision made for it. The same cut now looks **curved**.

Circled points are mistakes. They are on the wrong side in picture 2 *and* in picture 3.

In [ ]:
import torch, torch.nn as nn
from sklearn.datasets import make_moons, make_circles

activation = nn.Tanh()          # try nn.ReLU(), nn.Sigmoid(), nn.SiLU() (swish), nn.GELU(), nn.Identity()
X, y = make_moons(400, noise=0.08, random_state=0)   # later: make_circles(400, noise=0.05, factor=0.4)
X = (X - X.mean(0)) / X.std(0)

torch.manual_seed(6)            # try other seeds
hidden = nn.Sequential(nn.Linear(2, 2), activation)
output = nn.Linear(2, 1)
net = nn.Sequential(hidden, output)

Xt, yt = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)[:, None]
opt = torch.optim.Adam(net.parameters(), lr=0.05, weight_decay=0.001)
for _ in range(1000):
    opt.zero_grad()
    nn.functional.binary_cross_entropy_with_logits(net(Xt), yt).backward()
    opt.step()

with torch.no_grad():
    f = lambda L: hidden(torch.tensor(L.T.copy(), dtype=torch.float32)).detach().numpy().T
    H = f(X.T)
    score = net(torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)).numpy().reshape(xx.shape)
    acc = ((net(Xt) > 0) == yt.bool()).float().mean().item()
    w_out, b_out = output.weight[0].numpy(), output.bias.item()

wrong = (score_data := w_out @ H + b_out > 0) != y

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

def points(ax, P):
    ax.scatter(*P, c=y, cmap="coolwarm", s=12, zorder=3)
    ax.scatter(*P[:, wrong], s=70, facecolors="none", edgecolors="k", zorder=4)

draw_grid(axes[0], lambda L: L, "1. input x")
points(axes[0], X.T)

draw_grid(axes[1], f, "2. hidden space h: straight cut")
points(axes[1], H)
pad = .3
lo = np.minimum(H.min(1), f(np.stack(lines).transpose(1, 0, 2).reshape(2, -1)).min(1)) - pad
hi = np.maximum(H.max(1), f(np.stack(lines).transpose(1, 0, 2).reshape(2, -1)).max(1)) + pad
hx, hy = np.meshgrid(np.linspace(lo[0], hi[0], 200), np.linspace(lo[1], hi[1], 200))
hs = w_out[0]*hx + w_out[1]*hy + b_out

square(axes[2], f"3. back in input space: curved cut ({acc:.0%} correct)")
points(axes[2], X.T)

for ax, (gx, gy, gs) in zip(axes[1:], [(hx, hy, hs), (xx, yy, score)]):
    ax.contourf(gx, gy, gs, [-1e9, 0, 1e9], colors=["tab:blue", "tab:red"], alpha=.15)
    ax.contour(gx, gy, gs, [0], colors="k", linewidths=3)

for ax in (axes[0], axes[2]):
    ax.set(xlim=(-2.5, 2.5), ylim=(-2.5, 2.5), xlabel="x1", ylabel="x2")
axes[1].set(xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), aspect="auto",
            xlabel="h1 (hidden neuron 1)", ylabel="h2 (hidden neuron 2)")
plt.tight_layout(); plt.show()

## Your turn

1. Use `nn.Identity()`. Why is the boundary straight again, no matter how long you train?
2. Compare `Tanh` and `ReLU`: what shape does the hidden space take, and what shape does the boundary take?
3. Moons stays around 90%, try a few seeds. Then switch to circles. Two hidden neurons struggle. Why? What could a third dimension do?
4. Back in WK3 with `[5, 5]`: what happens in each layer, in your own words? (You cannot draw 5D, but the steps are the same.)